# Data Collection
**Purpose:** Load Delhi's daily pollution data, fetch matching weather data, and merge them.

**Inputs:** `data/raw/city_day.csv` (CPCB data via Kaggle), Open-Meteo historical weather API

**Outputs:** `data/raw/weather_delhi.csv`, `data/cleaned/delhi_merged.csv`

## 1. Load pollution data and filter Delhi
All 26 cities are in one file. Delhi should have 2,009 rows, one for every day from 2015-01-01 to 2020-07-01.

In [4]:
from pathlib import Path
import pandas as pd

BASE = Path.cwd().parent          # D:\Air Pollution Regime
RAW = BASE / "data" / "raw"

air = pd.read_csv(RAW / "city_day.csv", parse_dates=["Date"])
delhi = air[air["City"] == "Delhi"].copy()
print(delhi.shape)                # expect (2009, 16)

(2009, 16)


## 2. Fetch daily weather from Open-Meteo
Daily temperature, rainfall, wind speed and wind direction for Delhi's coordinates and the same date range. Pollution files contain no weather, and weather explains why pollutant levels differ between similar-looking days.


In [5]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 28.61,
    "longitude": 77.21,
    "start_date": "2015-01-01",
    "end_date": "2020-07-01",
    "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant",
    "timezone": "Asia/Kolkata",
}
resp = requests.get(url, params=params, timeout=60)
resp.raise_for_status()
weather = pd.DataFrame(resp.json()["daily"]).rename(columns={"time": "Date"})
weather["Date"] = pd.to_datetime(weather["Date"])
weather.to_csv(RAW / "weather_delhi.csv", index=False)
print(weather.shape)              # expect (2009, 5)
print(weather.head())

(2009, 5)
        Date  temperature_2m_mean  precipitation_sum  wind_speed_10m_max  \
0 2015-01-01                 16.3                0.0                 8.9   
1 2015-01-02                 15.2                4.2                11.2   
2 2015-01-03                 15.6                0.1                 9.0   
3 2015-01-04                 14.4                0.0                13.5   
4 2015-01-05                 14.3                0.0                27.0   

   wind_direction_10m_dominant  
0                           66  
1                           73  
2                          312  
3                          317  
4                          302  


## 3. Merge pollution and weather on Date
Check: the row count must stay 2,009 and the weather columns must have no missing values.

In [6]:
df = delhi.merge(weather, on="Date", how="left")
print(delhi.shape, df.shape)      # row count must stay 2009
print(df[["temperature_2m_mean","precipitation_sum","wind_speed_10m_max"]].isna().sum())

(2009, 16) (2009, 20)
temperature_2m_mean    0
precipitation_sum      0
wind_speed_10m_max     0
dtype: int64


## 4. Save merged dataset


In [7]:
(BASE / "data" / "cleaned").mkdir(parents=True, exist_ok=True)
df.to_csv(BASE / "data" / "cleaned" / "delhi_merged.csv", index=False)